# Deifferent use of Cross Validation

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error



In [3]:
df = pd.read_csv(r'C:\Users\matti\OneDrive\Desktop\Notebooks_statistic_course\DATA\Advertising.csv')

X = df.drop('sales', axis=1)
y = df['sales']

In [4]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

In [5]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor


# 1 Model Selection

Cross Validation can support the selection of appropriate models by comparing their performances

In [6]:
models = {
    'Linear Regression': LinearRegression(),

    'Polynomial Regression': make_pipeline(
        PolynomialFeatures(degree=2),
        LinearRegression()
    ),

    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        random_state=101
    )
}

The above code defines three regression models to compare their performance on the same dataset. 

*Linear Regression* is normally used as a baseline model to identify linear relationships between the input features and the target variable.

*Polynomial Regression* allows the model to capture potential non-linear relationships. The make_pipeline() function combines feature transformation and model training into a single workflow. 

*Random Forest* uses an ensemble of 100 decision trees to identify more complex patterns in the data. The random_state=101 parameter ensures reproducibility of the random processes used by the Random Forest model. 

Comparing these three approaches helps evaluate whether a simple linear model, a polynomial model, or an ensemble-based method provides better predictive performance.

In [ ]:
cv_results = {} 

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='neg_mean_squared_error'
    )

    rmse = np.sqrt(-scores.mean())

    cv_results[name] = rmse



for model, rmse in cv_results.items():
    print(f'{model}: RMSE = {rmse:.3f}')

Linear Regression: RMSE = 1.753
Polynomial Regression: RMSE = 0.662
Random Forest: RMSE = 0.831


The cv_results dictionary stores the RMSE (Root Mean Squared Error) calculated for each model.

The for loop iterates through the models dictionary, retrieving each model's name and corresponding estimator. 

The cross_val_score() function then evaluates the model using five folds, splitting the dataset into five parts. 

In each iteration, four folds are used to train the model and the remaining fold is used for validation. 

This process is repeated five times so that each fold is used once for validation.

The parameter scoring='neg_mean_squared_error' specifies Mean Squared Error as the evaluation metric. Scikit-learn returns the negative value because its scoring system is designed so that higher scores are better. Therefore, -scores.mean() converts the average negative MSE into a positive MSE, and np.sqrt() calculates the square root to obtain the RMSE. 

RMSE measures the average magnitude of prediction errors, giving greater weight to larger errors.

The resulting RMSE is stored in the cv_results dictionary for each model. 

Finally, the last loop prints the RMSE values to three decimal places, making it easier to compare the models. A lower RMSE indicates better predictive performance, provided that all models are evaluated on the same dataset and using the same cross-validation strategy.

### Results Comparison

The cross-validation results show that Polynomial Regression achieved the lowest RMSE of 0.662, indicating the best predictive performance among the three models. 

Random Forest achieved an RMSE of 0.831, also performing considerably better than Linear Regression, which recorded the highest RMSE of 1.753. 

These results suggest that the relationship between the input features and the target variable may not be entirely linear, as both Polynomial Regression and Random Forest improved on the linear baseline. 


# 2 Hyperparameters Tuning

Cross Validation could be useful to select the right hyperparameter value as for the below case, supporting the selection of appropriate degreee in a polynomial regession.

In [8]:
degrees = range(1, 11)

cv_scores = {}


for d in degrees:

  pipeline = make_pipeline( PolynomialFeatures(degree=d),
                           LinearRegression() )

  scores = cross_val_score(
      pipeline, X, y, cv=5, scoring='neg_mean_squared_error' )

  rmse = np.sqrt(-scores.mean())

  cv_scores[d] = rmse



The range(1, 11) function generates degrees from 1 to 10, allowing the evaluation of models with increasing complexity. 

The cv_scores dictionary is initialised to store the RMSE obtained for each degree.

The for loop iterates through each degree and creates a pipeline using make_pipeline(). 

The PolynomialFeatures(degree=d) transformer generates polynomial features up to the selected degree, while LinearRegression() fits a regression model using these transformed features. 

Using a pipeline ensures that feature transformation and model training are performed together during cross-validation.

The cross_val_score() function evaluates each pipeline using 5-fold cross-validation and negative Mean Squared Error as the scoring metric. 

The average score is converted into a positive RMSE by multiplying it by -1 and calculating its square root. 

Finally, the RMSE is stored in cv_scores, with the polynomial degree used as the dictionary key.



In [9]:
best_degree = min( cv_scores, key=cv_scores.get )

print(f"Best degree: {best_degree}")

Best degree: 3


This code identifies the optimal polynomial degree based on the cross-validation results obtained in the previous step. 

The min() function searches the cv_scores dictionary for the key associated with the lowest RMSE value. 

The parameter key=cv_scores.get ensures that the comparison is based on the RMSE values rather than the polynomial degrees themselves.

The variable best_degree stores the degree that achieved the lowest average cross-validation RMSE. Finally, the print() statement displays the selected degree.

### Results Analysis

In this case, the result is degree 3, indicating that Polynomial Regression with third-degree polynomial features achieved the lowest RMSE among the degrees tested, from 1 to 10.

# 3 Estimating Generalisation Performance

Comparing training performances over multiple fold can help in understanding model generalising capacity.

In [10]:
scores = cross_val_score( models['Random Forest'],
                         X, y, cv=5,
                          scoring='neg_mean_squared_error' )

This code evaluates the Random Forest model using 5-fold cross-validation to assess its predictive performance. 

The cross_val_score() function trains and validates the model across five different data splits, using four folds for training and one fold for validation in each iteration. 

This process is repeated until every fold has been used for validation once.

The model is retrieved from the models dictionary using the key 'Random Forest'. 

The parameters X and y represent the input features and target variable, respectively, while cv=5 specifies the number of cross-validation folds. 

The parameter scoring='neg_mean_squared_error' defines Mean Squared Error as the evaluation metric. Scikit-learn returns the negative MSE because its scoring convention treats higher scores as better.

In [11]:
for i, score in enumerate(scores):
    print(f'Fold {i + 1}: {score}')

Fold 1: -0.539742099999999
Fold 2: -0.4487298749999981
Fold 3: -0.7878823999999988
Fold 4: -1.161377700000002
Fold 5: -0.5113474499999933


Each score represents the model's prediction error on a different validation subset. Since the scoring parameter was set to neg_mean_squared_error, all values are negative.

# Results Analysis

The results show that the model's performance varies across the five folds. Fold 2 has the lowest prediction error, with an MSE of approximately 0.449, while Fold 4 has the highest, with an MSE of approximately 1.161. This variation suggests that the model performs differently across the validation subsets, potentially because of differences in the observations or patterns within each subset.

I can calculate the average and convert it to RMSE:

In [12]:
mean_mse = -scores.mean()

rmse = np.sqrt(mean_mse)

print(f"Cross-validated RMSE: {rmse:.3f}")

Cross-validated RMSE: 0.831


In [13]:
# I can also examine the variability between the folds:

rmse_scores = np.sqrt(-scores)

print("RMSE for each fold:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Standard deviation:", rmse_scores.std())

RMSE for each fold: [0.73467142 0.66987303 0.8876274  1.07767235 0.71508562]
Mean RMSE: 0.8169859657047175
Standard deviation: 0.14949861819270605


The mean gives the average performance across the folds, while the standard deviation gives an indication of how much the performance varies between different folds.